In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
IMG_SIZE = 64
BATCH_SIZE = 8
NUM_CLASSES = 10
EPOCHS = 5   # keep small for safety

tf.keras.backend.clear_session()

# =========================
# LOAD DATA (NO RAM OVERLOAD)
# =========================
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

y_train = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_test  = keras.utils.to_categorical(y_test, NUM_CLASSES)

# =========================
# TF.DATA PIPELINE (IMPORTANT)
# =========================
def preprocess(x, y):
    x = tf.image.resize(x, (IMG_SIZE, IMG_SIZE))
    x = preprocess_input(x)
    return x, y

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))
val_ds = val_ds.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# =========================
# MODEL BUILDER
# =========================
def build_model(trainable=False):
    base = MobileNetV2(weights='imagenet', include_top=False,
                       input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = trainable

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    output = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    return keras.Model(base.input, output), base

# =========================
# PROBLEM 1 (FROZEN)
# =========================
print("=== Problem 1 ===")

model, base = build_model(trainable=False)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_frozen = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# =========================
# PROBLEM 2 (FINE-TUNE)
# =========================
print("=== Problem 2 ===")

base.trainable = True

# unfreeze last few layers only
for layer in base.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# =========================
# PROBLEM 3 (ABLATION)
# =========================
print("=== Problem 3 ===")

def run_ablation(n):
    tf.keras.backend.clear_session()
    m, b = build_model(trainable=True)

    for layer in b.layers[:-n]:
        layer.trainable = False

    m.compile(
        optimizer=keras.optimizers.Adam(1e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    h = m.fit(train_ds, validation_data=val_ds, epochs=3, verbose=0)
    return max(h.history['val_accuracy'])

print("Top 5 layers:", run_ablation(5))
print("Top 10 layers:", run_ablation(10))
print("All layers:", run_ablation(len(base.layers)))

# =========================
# PROBLEM 4 (SCRATCH CNN)
# =========================
print("=== Problem 4 ===")

scratch_model = keras.Sequential([
    layers.Input((IMG_SIZE, IMG_SIZE, 3)),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

scratch_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

scratch_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

# =========================
# DONE
# =========================
print("✅ Finished without crash")

=== Problem 1 ===


/tmp/ipykernel_17578/1089115783.py:46: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(weights='imagenet', include_top=False,


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 266s 41ms/step - accuracy: 0.6288 - loss: 1.0825 - val_accuracy: 0.6980 - val_loss: 0.8678
Epoch 2/5
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 270s 43ms/step - accuracy: 0.6861 - loss: 0.9084 - val_accuracy: 0.7055 - val_loss: 0.8556
Epoch 3/5
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 267s 43ms/step - accuracy: 0.7108 - loss: 0.8378 - val_accuracy: 0.7072 - val_loss: 0.8776
Epoch 4/5
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 256s 41ms/step - accuracy: 0.7258 - loss: 0.7851 - val_accuracy: 0.7129 - val_loss: 0.8698
Epoch 5/5
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 257s 41ms/step - accuracy: 0.7433 - loss: 0.7348 - val_accuracy: 0.7113 - val_loss: 0.9014
=== Problem 2 ===
Epoch 1/5
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 452s 70ms/step - accuracy: 0.5041 - loss: 2.0875 - val_accuracy: 0.6422 - val_loss: 1.2846
Epoch 2/5
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 434s 69ms/step - accuracy: 0.5990 - loss: 1.2259 - val_accuracy: 0.6717 - val_loss: 1.1387
Epoch 3/5

QUESTIONS:

1. Explain the concept of 'negative transfer'. Under what conditions
    might using ImageNet pre-trained weights hurt performance?

ANS.  Negative transfer occurs when knowledge from the source domain
    (ImageNet) actively degrades performance on the target domain
    rather than helping it.  It happens when:       
      • The source and target domains are fundamentally different
        in appearance — e.g., medical X-rays, satellite imagery, or
        depth maps.  ImageNet features encode natural RGB textures and
        object parts; these are irrelevant (even misleading) for
        single-channel or frequency-domain inputs.       
      • The target task requires very fine-grained local features that
        the coarse ImageNet representation suppresses.       
      • The pre-trained model is much larger than what the small target
        dataset can support, leading to a very dominant prior that
        cannot be corrected with limited fine-tuning data.         
  
  Concrete example: training a CNN to classify chest X-rays for
    pneumonia.  ImageNet features encode colour gradients, pet fur,
    food textures, etc. — none of which map to pathological patterns
    in greyscale radiographs.  Fine-tuning from ImageNet can
    initially push the model toward incorrect feature detectors,
    requiring more epochs to escape that basin than starting from
    random weights.

2. In the ablation study, unfreezing all layers leads to more
    overfitting.  Explain the bias-variance trade-off.
    Why do lower CNN layers generalise better?

ANS. When we unfreeze all layers, the number of trainable parameters
    jumps dramatically (14M+ for VGG16).  With only 45k CIFAR-10
    training samples, this high model capacity memorises training data
    → HIGH VARIANCE (overfitting).  Conversely, freezing everything
    means the model cannot adapt to CIFAR-10 at all → HIGH BIAS.
    Unfreezing only the top few layers strikes the right balance.      

Lower convolutional layers learn universal, low-level features —       

    Gabor-like edge detectors, colour blobs, orientation filters —
    that are statistically similar across almost all image domains.
    These features have low variance across datasets.  Upper layers
    encode task-specific, high-level semantics (dog faces, car
    wheels).  By keeping lower layers frozen we:
      • Preserve highly generalisable representations (low bias).
      • Reduce the number of parameters that need fitting (low
        variance for the given dataset size).
This is why unfreezing only the top 4-8 layers outperforms both
    full freezing and full unfreezing on CIFAR-10.

Q3. Beyond accuracy, what factors would influence model choice for
    a mobile deployment?  Name at least three.

    1. Model Size / Memory Footprint
       Mobile devices have limited RAM (typically 2-4 GB shared).
       VGG16 is ~528 MB; MobileNetV2 is ~14 MB.  A smaller model
       is essential for on-device inference.

    3. Battery & Thermal Constraints
       Continuous use of a heavy model drains the battery and causes
       thermal throttling, reducing accuracy over time.  Efficient
       architectures are critical for sustained mobile usage.

    5. Privacy / Offline Requirement
       If inference must happen on-device without a server round-trip
       (e.g., health apps, offline environments), the model must
       fit entirely on the device, ruling out very large architectures
       regardless of accuracy.

Q4. Suppose you have a completely new medical imaging dataset (X-ray scans, grayscale, 512×512) with
only 500 labelled training examples. Write a step-by-step transfer learning strategy you would follow,
justifying every choice (which base model, how many layers to freeze, learning rate, augmentation, etc.).

ANS. 1. Base Model      
Use DenseNet121 pre-trained on ImageNet (or CheXNet if available). It's parameter-efficient and dominates medical imaging benchmarks. Avoid VGG16 — 138M parameters will overfit 500 samples instantly.

2. Handle Grayscale Input    
Replicate the single channel 3 times → [x, x, x]. This keeps all pre-trained weights intact without any architectural surgery. Then apply the model's preprocess_input — skipping this corrupts all early gradients.

3. Resize       
Downscale 512×512 → 224×224 (model's native size). Saves memory and compute with no meaningful information loss for classification tasks.

4. Freeze Strategy         
Freeze ~85% of the network. With only 500 samples you simply don't have the statistical power to update millions of weights meaningfully — you'd be fitting noise. Unfreeze only the last dense block + your new head: GAP → Dense(256, ReLU) → Dropout(0.5) → Dense(N, softmax).

5. Learning Rate — Two Phases    

Phase 1 (head only, base fully frozen): lr = 1e-3 — head is randomly initialised so it needs a normal lr to reach a useful starting point
Phase 2 (last block unfrozen): drop to lr = 1e-5 — 100× smaller to nudge pre-trained features without overwriting them. Add ReduceLROnPlateau(patience=3) on top.

6. Augmentation —  Domain-Safe Only    
With only 500 samples, augmentation is mandatory — it's your primary defence against overfitting. The critical rule is that every transform must be medically plausible. Horizontal flips, small rotations (±10°), slight zoom (0.9–1.1×), and minor brightness/contrast shifts are all safe since they reflect real variation in patient positioning and scanner calibration. Low-level Gaussian noise mimics sensor variation across machines. Vertical flips and aggressive cutout are strictly off-limits — lungs don't appear upside-down, and hiding regions may occlude the exact diagnostic signal you're trying to learn. Colour jitter is irrelevant on grayscale. MixUp (α=0.2) is worth adding as a soft batch-level regulariser with no domain-specific downsides.

7. Regularisation      
Dropout 0.5 in head + L2 weight decay 1e-4 on unfrozen layers + EarlyStopping(patience=10, restore_best_weights=True). Use patience=10 not 5 — val curves are noisy with tiny datasets.

8. Validation & Metric       
80/20 split gives only 100 val samples — too noisy. Use 5-fold cross-validation instead. Report AUC-ROC, not accuracy — medical datasets are class-imbalanced and accuracy is misleading. Use weighted cross-entropy loss to handle the imbalance at training time.